In [ ]:
!pip install torch torchvision timm tqdm --quiet

In [ ]:
import torch
from torchvision.datasets import Food101
from torchvision import transforms
from torch.utils.data import DataLoader, Subset

# Load dataset
train_dataset = Food101(root="data", split="train", download=True)

In [ ]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

train_dataset.transform = transform

# Use subset
train_dataset = Subset(train_dataset, range(10000))

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [ ]:
from tqdm import tqdm
import torch.nn as nn
import torch.optim as optim

def train(model, loader, epochs=3):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    model.train()

    for epoch in range(epochs):
        total_loss = 0
        loop = tqdm(loader, desc=f"Epoch {epoch+1}/{epochs}")

        for images, labels in loop:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            loop.set_postfix(loss=loss.item())

        print(f"Epoch {epoch+1} Total Loss: {total_loss:.4f}")

In [ ]:
from torchvision.models import resnet50, ResNet50_Weights

weights = ResNet50_Weights.DEFAULT
resnet = resnet50(weights=weights)

resnet.fc = nn.Linear(resnet.fc.in_features, 101)
resnet = resnet.to(device)

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 136MB/s]


In [ ]:
train(resnet, train_loader, epochs=3)

Epoch 1/3: 100%|██████████| 313/313 [02:26<00:00,  2.14it/s, loss=1.04]


Epoch 1 Total Loss: 321.5891


Epoch 2/3: 100%|██████████| 313/313 [02:26<00:00,  2.14it/s, loss=1.16]


Epoch 2 Total Loss: 182.3027


Epoch 3/3: 100%|██████████| 313/313 [02:28<00:00,  2.11it/s, loss=0.276]

Epoch 3 Total Loss: 129.8287


In [ ]:
import timm

effnet = timm.create_model('efficientnet_b0', pretrained=True, num_classes=101)
effnet = effnet.to(device)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/21.4M [00:00<?, ?B/s]

In [ ]:
train(effnet, train_loader, epochs=3)

Epoch 1/3: 100%|██████████| 313/313 [01:32<00:00,  3.39it/s, loss=0.77]


Epoch 1 Total Loss: 267.8410


Epoch 2/3: 100%|██████████| 313/313 [01:32<00:00,  3.37it/s, loss=1.26]


Epoch 2 Total Loss: 121.7170


Epoch 3/3: 100%|██████████| 313/313 [01:32<00:00,  3.39it/s, loss=0.309]

Epoch 3 Total Loss: 77.4766


In [ ]:
def evaluate(model, loader):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            _, predicted = torch.max(outputs, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    return 100 * correct / total

In [ ]:
resnet_acc = evaluate(resnet, train_loader)
effnet_acc = evaluate(effnet, train_loader)

print("ResNet Accuracy:", resnet_acc)
print("EfficientNet Accuracy:", effnet_acc)

ResNet Accuracy: 91.5
EfficientNet Accuracy: 90.56
